<a href="https://colab.research.google.com/github/aniget/SoftUni-AI-Integrations-for-developers/blob/main/Vector%20Databases%2C%20Embeddings%20and%20RAG/RAG_video_transcript_analizer_with_ChromaDB_and_Anthropic_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
from pathlib import Path
import re

# Load the transcript file
path = Path("/content/Video_Transcript_AI_and_the_Future_of_Work.txt")
raw_text = path.read_text()

# Remove timestamps like [00:00.0 - 00:08.5]
cleaned = re.sub(r"\[\d{2}:\d{2}\.\d\s*-\s*\d{2}:\d{2}\.\d\]", "", raw_text)

# Normalize whitespace
cleaned = re.sub(r"\s+", " ", cleaned).strip()

cleaned[:500]

"Welcome, on behalf of the Stone Center on Socio-Economic Inequality. We're extremely pleased to be the co-host of the event this evening. Welcome to all of you here in the Prushansky Auditorium, and welcome to the 2,000 people on the live stream. We are extremely delighted to have these marvelous guests with us tonight. We're happy to welcome, first, Danielle Lee, I'm going to tell you about, the David Sarnoff Professor of Management of Technology at the MIT Sloan School of Management. Her recen"

In [20]:
!pip install -q "opentelemetry-sdk==1.21.0"
!pip install -q "opentelemetry-api==1.21.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-grpc 1.40.0 requires opentelemetry-sdk~=1.40.0, but you have opentelemetry-sdk 1.21.0 which is incompatible.
google-adk 1.26.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.21.0 which is incompatible.
google-adk 1.26.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.21.0 which is incompatible.
google-cloud-pubsub 2.35.0 requires opentelemetry-api>=1.27.0, but you have opentelemetry-api 1.21.0 which is incompatible.
google-cloud-pubsub 2.35.0 requires opentelemetry-sdk>=1.27.0, but you have opentelemetry-sdk 1.21.0 which is incompatible.
opentelemetry-exporter-gcp-trace 1.11.0 requires opentelemetry-api~=1.30, but you have opentelemetry-api 1.21.0 which is incompatible.
opentelemetry-exporter-gcp-trace 1.11.0 requires open

In [21]:
!pip install -q chromadb --upgrade
!pip install -q sentence_transformers
import chromadb
from sentence_transformers import SentenceTransformer

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.26.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.26.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have

In [22]:
!pip install -q nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [23]:
#Split transcript into sentences with NLTK
from nltk.tokenize import sent_tokenize
sentences = sent_tokenize(cleaned)

In [24]:
sentences[1:11]

["We're extremely pleased to be the co-host of the event this evening.",
 'Welcome to all of you here in the Prushansky Auditorium, and welcome to the 2,000 people on the live stream.',
 'We are extremely delighted to have these marvelous guests with us tonight.',
 "We're happy to welcome, first, Danielle Lee, I'm going to tell you about, the David Sarnoff Professor of Management of Technology at the MIT Sloan School of Management.",
 'Her recent work investigates how artificial intelligence and data analytics are transforming fundamental firm activities, everything from hiring and promotion to investments in research and development.',
 'Her research has been published in leading journals, including the Quarterly Journal of Economics, Science, and Management Science.',
 "Next to Danielle, we're really happy to have with us this evening, Daron Acemoglu.",
 "He is currently Institute Professor at MIT, and among many other roles, he serves as the Faculty Co-Director of MIT's Stone Center

In [32]:
def chunk_text_by_sentences(sentences, max_chunk_size=400, overlap_sentences=1):
    chunks = []
    current_chunk = []

    for sentence in sentences:
        # If adding this sentence would exceed the limit, finalize the chunk
        if sum(len(s) for s in current_chunk) + len(sentence) > max_chunk_size:
            chunks.append(' '.join(current_chunk))
            # Start a new chunk with overlap
            current_chunk = current_chunk[-overlap_sentences:]
        current_chunk.append(sentence)

    # Add the last chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

chunks = chunk_text_by_sentences(sentences)
len(chunks), chunks[0][:500]


(45,
 "Welcome, on behalf of the Stone Center on Socio-Economic Inequality. We're extremely pleased to be the co-host of the event this evening. Welcome to all of you here in the Prushansky Auditorium, and welcome to the 2,000 people on the live stream. We are extremely delighted to have these marvelous guests with us tonight.")

In [33]:

#Create Chroma client
chroma_client = chromadb.Client()

#create a collection
chroma_collection = chroma_client.get_or_create_collection(name="transcript")

#load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

#embed and add chunks
embeddings = model.encode(chunks).tolist()

chroma_collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Build a retrieval function

In [50]:
def retrieve_with_scores(query, k=2):
    query_embedding = model.encode([query]).tolist()[0]
    results = chroma_collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "distances"]
    )

    documents = results["documents"][0]
    distances = results["distances"][0]

    # Convert distances to similarities
    similarities = [1 - distance for distance in distances]
    return [
        {
            "text":document,
            "distance": distance,
            "similarity":similarity}
        for document, distance, similarity in zip(documents, distances, similarities)
    ]

In [52]:
retrieve_with_scores("which jobs are in risk of being replaced by AI")

[{'text': "In that case, I think the future for labor is not great, because automation, if it goes on by itself, is going to eliminate more and more jobs. But there is a different future where we use AI with a very different architecture towards being what I call pro-worker AI, and that could actually amplify worker capabilities. We can talk about it more later. I don't want to take too much time.",
  'distance': 0.68001788854599,
  'similarity': 0.31998211145401},
 {'text': "Everywhere one looks nowadays, there's talk about AI. So I wanted to ask each of you to talk for two minutes or so about what effects do you think AI will have on the labor market and on the future of work, and what are your hopes and worries about AI? And why don't we start with you, Duran? Thank you, Steve. It's great to be here.",
  'distance': 0.7360615134239197,
  'similarity': 0.2639384865760803}]

In [53]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 15.4 MB/s eta 0:00:00


In [59]:
from google.colab import userdata
from anthropic import Anthropic

# Retrieve the API key from Colab Secrets (set via the lock icon in the sidebar)
# Change the name of the CLAUDE api key accordingly
# First run you will have to provide access to that key
api_key = userdata.get('CLAUDEAI_API_KEY')
client = Anthropic(api_key=api_key)

In [74]:
def ask_transcript(question, k=3, similarity_threshold= 0.25):
  # Retrieve chunks
  retrieved = retrieve_with_scores(question, k)

  # Build context block
  contex = "\n\n\n".join([r["text"] for r in retrieved])

  # Bild prompt
  message = client.messages.create(
      model="claude-sonnet-4-20250514",
      system = """
                "You are a helpful assistant. "
                "Answer ONLY using the transcript context provided. "
                "If the answer is not in the transcript, say so."
      """ ,
      max_tokens=500,
      messages=[
          {"role": "user",
           "content": f"Transcript context: {contex}\n\nQuestion: {question}"},
          ]
  )

  return message.content[0].text

In [75]:
ask_transcript("What is the work of the future")

'Based on the transcript provided, the speaker doesn\'t directly define "the work of the future" but presents two possible scenarios:\n\n1. **Current trajectory**: If automation continues "by itself," the future for labor "is not great" because it will "eliminate more and more jobs." This can happen even without achieving true AGI, as automating tasks can occur even when not super effective.\n\n2. **Alternative future**: The speaker proposes using AI with "a very different architecture" toward what they call "pro-worker AI" that could "amplify worker capabilities." They suggest this approach would be "better for productivity, better for inequality, better for social cohesion."\n\nHowever, the speaker notes that the pro-worker AI approach is "unfortunately not so good for the business models of the big tech," implying that the current direction favors automation over worker amplification.\n\nThe transcript doesn\'t provide a comprehensive definition of what work will look like in the fu